In [1]:
import os

os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = (
    "python"  # avoid protobuf import issues
)

import pandas as pd
import numpy as np
import glob
import cv2
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------
# Load training metadata and images
train_imgs_path = "../input/petfinder-pawpularity-score/train"
train_csv = pd.read_csv("../input/petfinder-pawpularity-score/train.csv")

# Resize images to 224x224 and compute a simple image feature (mean intensity)
train_img_files = sorted(glob.glob(os.path.join(train_imgs_path, "*.jpg")))
train_img_means = []
for f in train_img_files:
    img = cv2.imread(f)
    if img is None:
        # fallback in case of a bad read
        train_img_means.append(0.0)
        continue
    img_resized = cv2.resize(img, (224, 224))
    train_img_means.append(img_resized.mean())
train_img_means = np.array(train_img_means).reshape(-1, 1)

# Prepare feature matrix: metadata (12 cols) + image mean (1 col)
meta_features = train_csv.drop(columns=["Id", "Pawpularity"]).values.astype(np.float32)
X = np.hstack([meta_features, train_img_means])
y = train_csv["Pawpularity"].values.astype(np.float32)



In [2]:
# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42)

# Train a Gradient Boosting Regressor (reasonable default params)
gbr = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42
)
gbr.fit(X_train, y_train)

# Evaluate on validation set
val_pred = gbr.predict(X_val)
val_rmse = mean_squared_error(y_val, val_pred, squared=False)
print(f"Validation RMSE: {val_rmse:.4f}")



Validation RMSE: 21.8615


In [3]:
# Load test metadata and compute the same image feature
test_csv = pd.read_csv("../input/petfinder-pawpularity-score/test.csv")
test_imgs_path = "../input/petfinder-pawpularity-score/test"
test_img_files = sorted(glob.glob(os.path.join(test_imgs_path, "*.jpg")))
test_img_means = []
for f in test_img_files:
    img = cv2.imread(f)
    if img is None:
        test_img_means.append(0.0)
        continue
    img_resized = cv2.resize(img, (224, 224))
    test_img_means.append(img_resized.mean())
test_img_means = np.array(test_img_means).reshape(-1, 1)

test_meta = test_csv.drop(columns=["Id"]).values.astype(np.float32)
X_test = np.hstack([test_meta, test_img_means])

# Predict Pawpularity for test set
test_pred = gbr.predict(X_test)

# Create submission DataFrame with required column names
submission = pd.DataFrame({"Id": test_csv["Id"], "Pawpularity": test_pred})

# Write to CSV with correct formatting
submission.to_csv("submission.csv", index=False, header=True)
print("Submission file 'submission.csv' created with shape:", submission.shape)

Submission file 'submission.csv' created with shape: (992, 2)
